# Splice Site Prediction Using Foundation Model Embeddings

**Pipeline Overview:**
1. Extract embeddings from foundation models (offline, one-time)
2. Train lightweight 3-class classifiers on embeddings
3. Evaluate using 5-fold cross-validation
4. Compare results across models and window sizes

**Expected duration:** ~3-4 hours total

**Models:** HyenaDNA, DNABert, NucleotideTransformer

**Window sizes:** 300, 600, 1000, 10000 bp

## Setup: Import Libraries and Configuration

In [1]:
import os
import sys
import json
import importlib
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Force-disable flash/triton attention kernels for compatibility
os.environ['FLASH_ATTENTION_DISABLE'] = '1'
os.environ['USE_FLASH_ATTENTION'] = '0'
os.environ['TRITON_DISABLE_LINE_INFO'] = '1'

# Add src to path
sys.path.insert(0, str(Path.cwd() / 'src'))

from config import *
import models
import splicing_embed_extract
importlib.reload(models)
importlib.reload(splicing_embed_extract)

from splicing_embed_extract import EmbeddingExtractor
from splicing_classifier import SpliceSiteClassifier
from splicing_dataset import EmbeddingDataset, create_embedding_dataloader
from splicing_metrics import MetricsComputer
from splicing_train import SpliceClassifierTrainer

print(f"✓ Imports successful (reloaded latest src modules)")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"Project directory: {PROJECT_DIR}")

✓ Imports successful (reloaded latest src modules)
Device: NVIDIA GeForce RTX 5080
Project directory: d:\Splice_FMs_seq_lengths


## Configuration Check

In [2]:
# Verify data files exist
print("Data files available:")
for ws in WINDOW_SIZES:
    gencode_dir = RAW_DATA_DIR / f'gencode{ws}.csv'
    gtex_dir = GTEX_DATA_DIR / f'gtex{ws}.csv'
    print(f"  Window {ws}:")
    print(f"    GENCODE: {gencode_dir.exists()}")
    print(f"    GTEx: {gtex_dir.exists()}")

print(f"\nOutput directories:")
print(f"  Embeddings: {EMBEDDINGS_DIR}")
print(f"  Results: {SPLICING_RESULTS_DIR}")
print(f"  TensorBoard: {SPLICING_TENSORBOARD_DIR}")

print(f"\nEmbedding extraction config:")
print(f"  Method: {EMBEDDING_CONFIG['method']}")
print(f"  Max length: {EMBEDDING_CONFIG['max_length']}")
print(f"  Use FP16: {EMBEDDING_CONFIG['use_fp16']} (GPU memory safety)")
print(f"  Adaptive batch sizes by window:")
for ws, bs in EMBEDDING_CONFIG.get('batch_size_by_window', {}).items():
    print(f"    Window {ws:5d} bp: batch_size = {bs}")

Data files available:
  Window 300:
    GENCODE: True
    GTEx: True
  Window 600:
    GENCODE: True
    GTEx: True
  Window 1000:
    GENCODE: True
    GTEx: True
  Window 10000:
    GENCODE: True
    GTEx: True

Output directories:
  Embeddings: d:\Splice_FMs_seq_lengths\data\embeddings
  Results: d:\Splice_FMs_seq_lengths\results\classifiers
  TensorBoard: d:\Splice_FMs_seq_lengths\logs\splicing_classifiers

Embedding extraction config:
  Method: center
  Max length: 10000
  Use FP16: True (GPU memory safety)
  Adaptive batch sizes by window:
    Window   300 bp: batch_size = 256
    Window   600 bp: batch_size = 128
    Window  1000 bp: batch_size = 64
    Window 10000 bp: batch_size = 4


## PHASE 1: Extract Embeddings (Optional - Run Once)

In [ ]:
# Skip this cell if embeddings are already extracted
EXTRACT_EMBEDDINGS = True  # Set to True to extract embeddings

if EXTRACT_EMBEDDINGS:
    print("Extracting embeddings from foundation models...")
    print("Configuration:")
    print(f"  Method: {EMBEDDING_CONFIG['method']}")
    print(f"  Max length: {EMBEDDING_CONFIG['max_length']}")
    print(f"  Use FP16: {EMBEDDING_CONFIG['use_fp16']}")
    print(f"  Device: {EMBEDDING_CONFIG['device']}")
    print(f"\nAdaptive batch sizes:")
    for ws, bs in EMBEDDING_CONFIG.get('batch_size_by_window', {}).items():
        print(f"  Window {ws:5d} bp: {bs:3d}")
    print(f"\nThis will take ~2-3 hours on RTX 5080\n")
    
    # Optional: run only DNABERT first for quick troubleshooting
    # MODELS_CONFIG = {'DNABert': MODELS_CONFIG['DNABert']}

    extractor = EmbeddingExtractor(device='cuda')
    stats = extractor.extract_all()
    
    print(f"\n✓ Embedding extraction completed")
    print(f"  Extracted: {stats['total_succeeded']}")
    print(f"  Skipped: {stats['total_skipped']}")
    print(f"  Failed: {stats['total_failed']}")
else:
    print("Skipping embedding extraction.")
    print("To extract embeddings, set EXTRACT_EMBEDDINGS = True and run this cell.")

## PHASE 2: List Available Embeddings

In [3]:
# Check which embeddings are available for CV training
available_combinations = []

for window_size in WINDOW_SIZES:
    for model_name in MODELS_CONFIG.keys():
        for model_id in MODELS_CONFIG[model_name]['model_ids']:
            combo_name = f"{model_name}_{model_id}"
            embed_dir = EMBEDDINGS_DIR / str(window_size) / combo_name
            
            trainval_file = embed_dir / "trainval_embeddings.pt"
            test_file = embed_dir / "test_embeddings.pt"
            gtex_file = embed_dir / "gtex_test_embeddings.pt"
            
            if trainval_file.exists() and test_file.exists() and gtex_file.exists():
                available_combinations.append({
                    'window_size': window_size,
                    'model_name': model_name,
                    'model_id': model_id,
                    'combo_name': combo_name,
                    'embed_dir': embed_dir
                })

print(f"Available embeddings for CV training: {len(available_combinations)}")
for combo in available_combinations:
    print(f"  ✓ Window {combo['window_size']:5d} | {combo['combo_name']:30s}")

if len(available_combinations) == 0:
    print("\n⚠ No trainval/test embeddings found. Please run embedding extraction first.")

Available embeddings for CV training: 16
  ✓ Window   300 | HyenaDNA_LongSafari/hyenadna-small-32k-seqlen-hf
  ✓ Window   300 | HyenaDNA_LongSafari/hyenadna-medium-160k-seqlen-hf
  ✓ Window   300 | DNABert_zhihan1996/DNABERT-2-117M
  ✓ Window   300 | NucleotideTransformer_InstaDeepAI/nucleotide-transformer-500m-human-ref
  ✓ Window   300 | NucleotideTransformer_InstaDeepAI/nucleotide-transformer-v2-100m-multi-species
  ✓ Window   600 | HyenaDNA_LongSafari/hyenadna-small-32k-seqlen-hf
  ✓ Window   600 | HyenaDNA_LongSafari/hyenadna-medium-160k-seqlen-hf
  ✓ Window   600 | DNABert_zhihan1996/DNABERT-2-117M
  ✓ Window   600 | NucleotideTransformer_InstaDeepAI/nucleotide-transformer-500m-human-ref
  ✓ Window   600 | NucleotideTransformer_InstaDeepAI/nucleotide-transformer-v2-100m-multi-species
  ✓ Window  1000 | HyenaDNA_LongSafari/hyenadna-small-32k-seqlen-hf
  ✓ Window  1000 | HyenaDNA_LongSafari/hyenadna-medium-160k-seqlen-hf
  ✓ Window  1000 | DNABert_zhihan1996/DNABERT-2-117M
  ✓ Wind

## PHASE 3: Train Classifiers with 5-Fold CV

In [4]:
# Train classifiers for each combination
all_results = {}
training_start_time = datetime.now()

# Skip training if result file already exists
SKIP_IF_RESULTS_EXIST = True

for idx, combo in enumerate(available_combinations):
    print(f"\n{'='*80}")
    print(f"[{idx+1}/{len(available_combinations)}] Training: {combo['combo_name']} (Window {combo['window_size']})")
    print(f"{'='*80}")
    
    experiment_name = f"{combo['combo_name']}_w{combo['window_size']}"
    results_dir = SPLICING_RESULTS_DIR / experiment_name
    results_file = results_dir / "results.json"

    # Skip if already trained, but still load into all_results for downstream summary
    if SKIP_IF_RESULTS_EXIST and results_file.exists():
        try:
            with open(results_file, 'r') as f:
                existing_results = json.load(f)
            all_results[experiment_name] = existing_results
            print(f"↷ Skipped (already trained): {experiment_name}")
            print(f"  Loaded existing results: {results_file}")
            continue
        except Exception as e:
            print(f"⚠ Found existing results but failed to load ({e}), retraining...")
    
    # Load train+val embeddings prepared for CV
    trainval_file = combo['embed_dir'] / "trainval_embeddings.pt"
    trainval_data = torch.load(trainval_file)
    
    all_embeddings = trainval_data['embeddings']
    all_labels = trainval_data['labels']
    
    embedding_dim = all_embeddings.shape[1]
    
    # Train with CV
    trainer = SpliceClassifierTrainer(
        embedding_dim=embedding_dim,
        device='cuda' if torch.cuda.is_available() else 'cpu',
        results_dir=str(SPLICING_RESULTS_DIR)
    )
    
    results = trainer.train_with_cv(
        all_embeddings.numpy(),
        all_labels.numpy(),
        experiment_name=experiment_name,
        num_folds=5,
        num_epochs=50,
        batch_size=256,
        learning_rate=0.0001,
        weight_decay=0.001,
        early_stopping_patience=5,
        seed=42,
        deterministic=False
    )
    
    all_results[experiment_name] = results
    
    print(f"✓ {experiment_name} training completed")

elapsed_hours = (datetime.now() - training_start_time).total_seconds() / 3600
print(f"\n{'='*80}")
print(f"Total training time: {elapsed_hours:.1f} hours")
print(f"Total experiments available in all_results: {len(all_results)}")
print(f"{'='*80}")

2026-03-05 13:20:18,850 - splicing_train - INFO - 
2026-03-05 13:20:18,850 - splicing_train - INFO - TRAINING: HyenaDNA_LongSafari/hyenadna-small-32k-seqlen-hf_w300
2026-03-05 13:20:18,850 - splicing_train - INFO - ================================================================================
2026-03-05 13:20:18,850 - splicing_train - INFO - Embeddings shape: (718417, 256)
2026-03-05 13:20:18,850 - splicing_train - INFO - Labels shape: (718417,)
2026-03-05 13:20:18,854 - splicing_train - INFO - Label distribution: [239327 242009 237081]
2026-03-05 13:20:18,854 - splicing_train - INFO - Num folds: 5, Epochs: 50, Batch size: 256
2026-03-05 13:20:18,908 - splicing_train - INFO - 
################################################################################



[1/16] Training: HyenaDNA_LongSafari/hyenadna-small-32k-seqlen-hf (Window 300)


2026-03-05 13:20:18,913 - splicing_train - INFO - FOLD 1/5
2026-03-05 13:20:18,913 - splicing_train - INFO - ################################################################################
2026-03-05 13:20:18,983 - splicing_train - INFO - Train: 574733, Val: 143684
2026-03-05 13:20:18,987 - splicing_train - INFO - Train label dist: [191461 193608 189664]
2026-03-05 13:20:18,989 - splicing_train - INFO - Val label dist: [47866 48401 47417]
2026-03-05 13:20:27,123 - splicing_train - INFO - Epoch   1: train_loss=0.5545, val_loss=0.5201, acc=0.7684, f1=0.7592, mcc=0.6625, auprc=0.8224 ✓ (best by mcc: 1)
2026-03-05 13:20:34,852 - splicing_train - INFO - Epoch   2: train_loss=0.5287, val_loss=0.5181, acc=0.7706, f1=0.7613, mcc=0.6657, auprc=0.8249 ✓ (best by mcc: 2)
2026-03-05 13:20:42,790 - splicing_train - INFO - Epoch   3: train_loss=0.5226, val_loss=0.5096, acc=0.7784, f1=0.7716, mcc=0.6744, auprc=0.8277 ✓ (best by mcc: 3)
2026-03-05 13:20:51,654 - splicing_train - INFO - Epoch   4: tra

✓ HyenaDNA_LongSafari/hyenadna-small-32k-seqlen-hf_w300 training completed

[2/16] Training: HyenaDNA_LongSafari/hyenadna-medium-160k-seqlen-hf (Window 300)


2026-03-05 13:29:07,596 - splicing_train - INFO - FOLD 1/5
2026-03-05 13:29:07,596 - splicing_train - INFO - ################################################################################
2026-03-05 13:29:07,692 - splicing_train - INFO - Train: 574733, Val: 143684
2026-03-05 13:29:07,694 - splicing_train - INFO - Train label dist: [191461 193608 189664]
2026-03-05 13:29:07,695 - splicing_train - INFO - Val label dist: [47866 48401 47417]
2026-03-05 13:29:15,773 - splicing_train - INFO - Epoch   1: train_loss=0.4978, val_loss=0.4726, acc=0.7927, f1=0.7887, mcc=0.6918, auprc=0.8629 ✓ (best by mcc: 1)
2026-03-05 13:29:23,548 - splicing_train - INFO - Epoch   2: train_loss=0.4789, val_loss=0.4684, acc=0.7944, f1=0.7906, mcc=0.6937, auprc=0.8656 ✓ (best by mcc: 2)
2026-03-05 13:29:31,298 - splicing_train - INFO - Epoch   3: train_loss=0.4749, val_loss=0.4667, acc=0.7954, f1=0.7933, mcc=0.6943, auprc=0.8667 ✓ (best by mcc: 3)
2026-03-05 13:29:39,084 - splicing_train - INFO - Epoch   4: tra

...

✓ NucleotideTransformer_InstaDeepAI/nucleotide-transformer-500m-human-ref_w1000 training completed

[15/16] Training: NucleotideTransformer_InstaDeepAI/nucleotide-transformer-v2-100m-multi-species (Window 1000)


2026-03-05 18:45:35,630 - splicing_train - INFO - 
2026-03-05 18:45:35,630 - splicing_train - INFO - TRAINING: NucleotideTransformer_InstaDeepAI/nucleotide-transformer-v2-100m-multi-species_w1000
2026-03-05 18:45:35,630 - splicing_train - INFO - ================================================================================
2026-03-05 18:45:35,630 - splicing_train - INFO - Embeddings shape: (718545, 512)
2026-03-05 18:45:35,630 - splicing_train - INFO - Labels shape: (718545,)
2026-03-05 18:45:35,630 - splicing_train - INFO - Label distribution: [239460 242008 237077]
2026-03-05 18:45:35,630 - splicing_train - INFO - Num folds: 5, Epochs: 50, Batch size: 256
2026-03-05 18:45:35,714 - splicing_train - INFO - 
################################################################################
2026-03-05 18:45:35,718 - splicing_train - INFO - FOLD 1/5
2026-03-05 18:45:35,718 - splicing_train - INFO - ################################################################################
2026-03-05

✓ NucleotideTransformer_InstaDeepAI/nucleotide-transformer-v2-100m-multi-species_w1000 training completed

[16/16] Training: HyenaDNA_LongSafari/hyenadna-small-32k-seqlen-hf (Window 10000)


2026-03-05 19:18:52,450 - splicing_train - INFO - 
################################################################################
2026-03-05 19:18:52,451 - splicing_train - INFO - FOLD 1/5
2026-03-05 19:18:52,451 - splicing_train - INFO - ################################################################################
2026-03-05 19:18:52,538 - splicing_train - INFO - Train: 574602, Val: 143651
2026-03-05 19:18:52,542 - splicing_train - INFO - Train label dist: [191458 193543 189601]
2026-03-05 19:18:52,543 - splicing_train - INFO - Val label dist: [47865 48386 47400]
2026-03-05 19:19:00,393 - splicing_train - INFO - Epoch   1: train_loss=0.6423, val_loss=0.6222, acc=0.7383, f1=0.7328, mcc=0.6103, auprc=0.7493 ✓ (best by mcc: 1)
2026-03-05 19:19:08,197 - splicing_train - INFO - Epoch   2: train_loss=0.6260, val_loss=0.6199, acc=0.7382, f1=0.7326, mcc=0.6102, auprc=0.7534 (patience: 1/5)
2026-03-05 19:19:15,987 - splicing_train - INFO - Epoch   3: train_loss=0.6229, val_loss=0.6189, ac

✓ HyenaDNA_LongSafari/hyenadna-small-32k-seqlen-hf_w10000 training completed

Total training time: 6.3 hours
Total experiments available in all_results: 16


## PHASE 4: Results Comparison and Analysis

In [5]:
# Summarize results + export aggregate files
summary_data = []

for exp_name, results in all_results.items():
    avg_metrics = results['averaged_metrics']
    
    # Parse window size from experiment name suffix: ..._w300
    window_size = None
    if '_w' in exp_name:
        try:
            window_size = int(exp_name.rsplit('_w', 1)[-1])
        except ValueError:
            window_size = None

    summary_data.append({
        'Experiment': exp_name,
        'Window': window_size,
        'Accuracy': avg_metrics.get('accuracy_mean', 0),
        'Balanced Acc': avg_metrics.get('balanced_accuracy_mean', 0),
        'F1 (weighted)': avg_metrics.get('f1_weighted_mean', 0),
        'F1 (macro)': avg_metrics.get('f1_macro_mean', 0),
        'MCC': avg_metrics.get('mcc_mean', 0),
        'ROC-AUC': avg_metrics.get('roc_auc_weighted_mean', 0),
    })

summary_df = pd.DataFrame(summary_data).sort_values('F1 (weighted)', ascending=False)

print("\n" + "="*100)
print("RESULTS SUMMARY (Sorted by F1-Weighted)")
print("="*100)
print(summary_df.to_string(index=False))
print("="*100)

# Export aggregate summaries
summary_dir = SPLICING_RESULTS_DIR / 'summaries'
summary_dir.mkdir(parents=True, exist_ok=True)

all_csv = summary_dir / 'all_experiments_summary.csv'
all_json = summary_dir / 'all_experiments_summary.json'
summary_df.to_csv(all_csv, index=False)
summary_df.to_json(all_json, orient='records', indent=2)

print(f"\nSaved aggregate summaries:")
print(f"  - {all_csv}")
print(f"  - {all_json}")

# Export one file per window
windows_exported = []
if 'Window' in summary_df.columns:
    for ws in sorted([w for w in summary_df['Window'].dropna().unique()]):
        ws_int = int(ws)
        ws_df = summary_df[summary_df['Window'] == ws].copy().sort_values('F1 (weighted)', ascending=False)
        ws_csv = summary_dir / f'summary_w{ws_int}.csv'
        ws_json = summary_dir / f'summary_w{ws_int}.json'
        ws_df.to_csv(ws_csv, index=False)
        ws_df.to_json(ws_json, orient='records', indent=2)
        windows_exported.append((ws_csv, ws_json))

if windows_exported:
    print("\nSaved per-window summaries:")
    for ws_csv, ws_json in windows_exported:
        print(f"  - {ws_csv}")
        print(f"  - {ws_json}")


RESULTS SUMMARY (Sorted by F1-Weighted)
                                                                          Experiment  Window  Accuracy  Balanced Acc  F1 (weighted)  F1 (macro)      MCC  ROC-AUC
        NucleotideTransformer_InstaDeepAI/nucleotide-transformer-500m-human-ref_w600     600  0.919324      0.919305       0.919226    0.919169 0.879048 0.985398
        NucleotideTransformer_InstaDeepAI/nucleotide-transformer-500m-human-ref_w300     300  0.909771      0.909743       0.909545    0.909466 0.864782 0.981873
       NucleotideTransformer_InstaDeepAI/nucleotide-transformer-500m-human-ref_w1000    1000  0.890775      0.890805       0.890460    0.890473 0.836253 0.975664
                             HyenaDNA_LongSafari/hyenadna-medium-160k-seqlen-hf_w600     600  0.822269      0.822350       0.820227    0.820250 0.734296 0.943489
                            HyenaDNA_LongSafari/hyenadna-medium-160k-seqlen-hf_w1000    1000  0.809408      0.809514       0.806453    0.806447 0.715

## PHASE 5: Best Model Evaluation on Test Set

In [ ]:
# Evaluate best model on test sets
print("\nEvaluating best models on GENCODE and GTEx test sets...\n")

test_results = {}

for combo in available_combinations:
    experiment_name = f"{combo['combo_name']}_w{combo['window_size']}"
    results_dir = SPLICING_RESULTS_DIR / experiment_name
    results_file = results_dir / 'results.json'
    
    if not results_file.exists():
        print(f"⚠ Results file not found: {results_file}")
        continue
    
    with open(results_file) as f:
        cv_results = json.load(f)
    
    # Get best epoch from fold 0
    best_epoch = cv_results['per_fold_results']['fold_0']['best_epoch']
    
    print(f"✓ {experiment_name}: Best epoch = {best_epoch}")
    test_results[experiment_name] = cv_results

print(f"\n✓ Loaded results for {len(test_results)} experiments")

## Summary

In [ ]:
print("\n" + "="*80)
print("PIPELINE SUMMARY")
print("="*80)

print(f"\n✓ Embedding extraction: {len(available_combinations)} combinations")
print(f"✓ Classifier training: 5-fold CV with early stopping")
print(f"✓ Results saved:")
print(f"  - JSON files: {SPLICING_RESULTS_DIR}")
print(f"  - TensorBoard logs: {SPLICING_TENSORBOARD_DIR}")

print(f"\nBest performing model (F1-weighted):")
best_row = summary_df.iloc[0]
print(f"  {best_row['Experiment']}")
print(f"    F1-weighted: {best_row['F1 (weighted)']:.4f}")
print(f"    Accuracy: {best_row['Accuracy']:.4f}")
print(f"    MCC: {best_row['MCC']:.4f}")

print(f"\nView TensorBoard results:")
print(f"  tensorboard --logdir {SPLICING_TENSORBOARD_DIR}")
print(f"\n" + "="*80)